# Module 9: Your First Active Inference Agent

**Following Smith, Friston & Whyte (2022) step-by-step**

---

In this notebook we build a complete Active Inference agent from scratch and run it on the
**T-maze** -- the canonical benchmark introduced by Friston and colleagues. By the end you
will understand:

1. **The POMDP formulation**: A (likelihood), B (transitions), C (preferences), D (priors)
2. **What each matrix means**: A = "how do I observe states?", B = "how do states change?",
   C = "what do I want to observe?", D = "what do I believe initially?"
3. **Belief updating via belief propagation** (message passing on a factor graph)
4. **EFE-based action selection**: $\pi(a) \propto \exp(-\gamma \cdot G(a))$
5. **The T-maze**: the canonical Active Inference benchmark

### References

- Smith, Friston & Whyte (2022). *A Step-by-Step Tutorial on Active Inference*.
  Journal of Mathematical Psychology, 107, 102632.
- Da Costa et al. (2020). *Active Inference on Discrete State-Spaces: A Synthesis*.
  Journal of Mathematical Psychology, 99, 102447.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# PGMax Active Inference modules
from pgmax.aif.generative_model import GenerativeModel
from pgmax.aif.agent import ActiveInferenceAgent
from pgmax.aif.inference import infer_states
from pgmax.aif.policy import evaluate_policies, select_action
from pgmax.aif.benchmarks.t_maze import (
    build_t_maze_model, TMazeEnv, run_t_maze,
    STATE_NAMES, OBS_NAMES, ACTION_NAMES,
    ACT_STAY, ACT_CUE, ACT_LEFT, ACT_RIGHT,
    NUM_STATES, NUM_OBS, NUM_ACTIONS,
)

# Curriculum plotting utilities
import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_belief_evolution, plot_efe_decomposition,
    plot_matrix_heatmap, plot_policy_distribution, dual_perspective_box, COLORS
)

np.set_printoptions(precision=3, suppress=True)
print("All imports successful.")

## 1. The T-Maze: Our Testing Ground

The T-maze encodes a simple decision problem:

```
    [LEFT arm]  ----  [CENTER]  ----  [RIGHT arm]
                          |
                       [CUE]
```

- The agent starts at **center** and doesn't know which arm is rewarded.
- A **cue** at the bottom reveals (with some reliability) which arm has the reward.
- The optimal strategy: **visit the cue first**, then go to the correct arm.

### State space

We encode both **location** and **reward condition** into a single state factor
with 8 states: (location) x (reward-left, reward-right).

In [ ]:
# ── Step 1: Build the A matrix (likelihood) manually ──────────────────────
#
# A[o, s] = P(observation=o | state=s)
# Shape: (5 observations, 8 states)
#
# The A matrix answers: "If the world is in state s, what will I observe?"

cue_reliability = 0.9

A = np.zeros((NUM_OBS, NUM_STATES))

# At center (states 0,1): always observe "null" -- no information
A[0, 0] = 1.0  # center/reward-left  -> null
A[0, 1] = 1.0  # center/reward-right -> null

# At cue (states 2,3): observe cue with reliability
A[1, 2] = cue_reliability       # cue/reward-left  -> cue_left (correct)
A[2, 2] = 1.0 - cue_reliability # cue/reward-left  -> cue_right (error)
A[2, 3] = cue_reliability       # cue/reward-right -> cue_right (correct)
A[1, 3] = 1.0 - cue_reliability # cue/reward-right -> cue_left (error)

# At left arm (states 4,5):
A[3, 4] = 1.0  # left/reward-left  -> REWARD
A[4, 5] = 1.0  # left/reward-right -> PUNISHMENT

# At right arm (states 6,7):
A[4, 6] = 1.0  # right/reward-left  -> PUNISHMENT
A[3, 7] = 1.0  # right/reward-right -> REWARD

print("A matrix (likelihood) -- P(observation | state):")
print(f"  Rows: {OBS_NAMES}")
print(f"  Cols: {STATE_NAMES}")
print()
print(A)
print()
print("Each column sums to 1:", A.sum(axis=0))

In [ ]:
# ── Step 2: Visualize the A matrix as a heatmap ──────────────────────────
#
# This reveals the observation structure:
# - Center states produce only "null" observations
# - Cue states are informative (but noisy) about the reward condition
# - Arm states produce deterministic reward/punishment

fig, ax = plt.subplots(figsize=(10, 5))
plot_matrix_heatmap(
    A,
    title="A Matrix: How Observations Map to States -- P(o|s)",
    row_labels=OBS_NAMES,
    col_labels=STATE_NAMES,
    cmap="YlOrRd",
    ax=ax,
)
ax.set_xlabel("Hidden State")
ax.set_ylabel("Observation")
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 3: Build the B matrix (transitions) ─────────────────────────────
#
# B[s', s, a] = P(next_state=s' | current_state=s, action=a)
# Shape: (8 states, 8 states, 4 actions)
#
# The B matrix answers: "If I'm in state s and take action a, where will I end up?"
# Key constraint: the reward condition NEVER changes (it's a property of the trial).

B = np.zeros((NUM_STATES, NUM_STATES, NUM_ACTIONS))

for reward_side in range(2):  # 0=left, 1=right
    center = reward_side       # 0 or 1
    cue = 2 + reward_side     # 2 or 3
    left = 4 + reward_side    # 4 or 5
    right = 6 + reward_side   # 6 or 7

    # ACT_STAY (0): remain at current location
    for s in [center, cue, left, right]:
        B[s, s, ACT_STAY] = 1.0

    # ACT_CUE (1): center -> cue, others stay
    B[cue, center, ACT_CUE] = 1.0
    B[cue, cue, ACT_CUE] = 1.0
    B[left, left, ACT_CUE] = 1.0
    B[right, right, ACT_CUE] = 1.0

    # ACT_LEFT (2): center/cue -> left arm
    B[left, center, ACT_LEFT] = 1.0
    B[left, cue, ACT_LEFT] = 1.0
    B[left, left, ACT_LEFT] = 1.0
    B[right, right, ACT_LEFT] = 1.0  # can't switch arms

    # ACT_RIGHT (3): center/cue -> right arm
    B[right, center, ACT_RIGHT] = 1.0
    B[right, cue, ACT_RIGHT] = 1.0
    B[right, right, ACT_RIGHT] = 1.0
    B[left, left, ACT_RIGHT] = 1.0   # can't switch arms

print("B matrix shape:", B.shape)
print("Each B[:,:,a] column sums to 1:")
for a in range(NUM_ACTIONS):
    print(f"  {ACTION_NAMES[a]:10s}: {B[:, :, a].sum(axis=0)}")

In [ ]:
# ── Visualize B matrices for each action ─────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for a_idx in range(NUM_ACTIONS):
    plot_matrix_heatmap(
        B[:, :, a_idx],
        title=f"B[:,:,{ACTION_NAMES[a_idx]}]",
        row_labels=STATE_NAMES,
        col_labels=STATE_NAMES,
        cmap="Blues",
        ax=axes[a_idx],
    )
    axes[a_idx].set_xlabel("Current State")
    axes[a_idx].set_ylabel("Next State")

fig.suptitle("B Matrices: How Actions Change the World -- P(s'|s, a)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 4: Define C (preferences) and D (priors) ────────────────────────

# C: Log-preferences over observations
# "What observations does the agent WANT to experience?"
reward_magnitude = 3.0
C = np.zeros(NUM_OBS)
C[0] = 0.0   # null: neutral
C[1] = 0.0   # cue_left: neutral (informative but not preferred per se)
C[2] = 0.0   # cue_right: neutral
C[3] = reward_magnitude   # reward: strongly preferred
C[4] = -reward_magnitude  # punishment: strongly avoided

print("C (preferences over observations):")
for name, val in zip(OBS_NAMES, C):
    bar = '+' * int(abs(val)) if val >= 0 else '-' * int(abs(val))
    print(f"  {name:12s}: {val:+.1f}  {bar}")

print()

# D: Prior beliefs over initial states
# "Where does the agent think it starts?"
# The agent knows it starts at center but doesn't know the reward side.
D = np.zeros(NUM_STATES)
D[0] = 0.5  # center/reward-left
D[1] = 0.5  # center/reward-right

print("D (prior beliefs):")
for name, val in zip(STATE_NAMES, D):
    print(f"  {name:12s}: {val:.2f}")

In [ ]:
# ── Step 5: Assemble the Generative Model ────────────────────────────────
#
# Now we package everything into a GenerativeModel. This is the POMDP that
# defines the agent's beliefs about how the world works.
#
# T=2 means the agent plans 2 steps ahead (visit cue, then choose arm).

gm = GenerativeModel(A=[A], B=[B], C=[C], D=[D], T=2)

print(f"Generative Model Summary:")
print(f"  Modalities:    {gm.num_modalities}")
print(f"  State factors: {gm.num_factors}")
print(f"  Observations:  {gm.num_obs}")
print(f"  States:        {gm.num_states}")
print(f"  Actions:       {gm.num_actions}")
print(f"  Planning T:    {gm.T}")
print(f"  Num policies:  {gm.num_policies}")
print()
print("Example policies (first 5):")
for i in range(min(5, gm.num_policies)):
    p = gm.policies[i]
    actions = [ACTION_NAMES[int(p[t, 0])] for t in range(gm.T)]
    print(f"  Policy {i:2d}: {' -> '.join(actions)}")

## 2. Running the Agent on the T-Maze

Now we create an `ActiveInferenceAgent` and run it on a single trial.
The agent will:
1. Observe the environment
2. Update its beliefs about the hidden state (via belief propagation)
3. Evaluate Expected Free Energy for all candidate policies
4. Select the action from the best policy

The AIF action selection rule:

$$P(\pi) = \sigma\big(-\gamma \cdot G(\pi) + \ln E(\pi)\big)$$

where $G(\pi)$ is the Expected Free Energy and $\gamma$ is the policy precision.

In [ ]:
# ── Step 6: Run a single trial ──────────────────────────────────────────

# Create agent and environment
agent = ActiveInferenceAgent(gm, gamma=4.0, seed=42)
env = TMazeEnv(reward_side="left", cue_reliability=0.9, seed=42)

# Reset
agent.reset()
obs = env.reset()

print("=== Single T-Maze Trial (reward is on the LEFT) ===")
print(f"\nInitial observation: {OBS_NAMES[obs]}")
print(f"Initial beliefs: {agent.beliefs[0]}")
print()

# Step through the trial
for step in range(2):
    print(f"--- Step {step + 1} ---")
    action, info = agent.step([obs])
    print(f"  Action chosen: {ACTION_NAMES[action]}")
    print(f"  Updated beliefs: {info['beliefs'][0].round(3)}")
    print(f"  Policy probs (top 3):")
    
    # Show top 3 policies
    probs = info['policy_probs']
    top_idx = np.argsort(probs)[::-1][:3]
    for idx in top_idx:
        p = gm.policies[idx]
        acts = [ACTION_NAMES[int(p[t, 0])] for t in range(gm.T)]
        print(f"    [{' -> '.join(acts)}]: {probs[idx]:.3f}")
    
    obs, reward, done = env.step(action)
    print(f"  Observation: {OBS_NAMES[obs]}")
    print(f"  Reward: {reward}")
    print(f"  Done: {done}")
    print()

print(f"Final location: {env.location}")
print(f"Outcome: {'SUCCESS' if reward > 0 else 'FAILURE'}")

In [ ]:
# ── Step 7: Plot belief evolution ────────────────────────────────────────
#
# Visualize how the agent's beliefs about the hidden state evolve
# as it receives observations.

belief_history = agent.belief_history

# Extract beliefs for factor 0 (the only factor)
beliefs_array = np.array([b[0] for b in belief_history])

fig, ax = plt.subplots(figsize=(12, 5))
plot_belief_evolution(
    beliefs_array,
    state_names=STATE_NAMES,
    title="Belief Evolution During T-Maze Trial",
    ax=ax,
)
ax.set_xlabel("Step")
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='Step 1 (observe null)')
ax.axvline(x=1, color='gray', linestyle=':', alpha=0.5, label='Step 2 (observe cue)')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- After the null observation: beliefs remain at prior (50/50 on reward side)")
print("- After visiting the cue: beliefs shift strongly toward the correct reward side")
print("  The agent now 'knows' where the reward is and can act accordingly.")

In [ ]:
# ── Step 8: Plot EFE for each policy ─────────────────────────────────────
#
# The EFE values explain WHY the agent prefers "go to cue first".
# Lower G = better policy.

G = agent.efe_history[0]  # EFE values from step 1

# Label each policy
policy_labels = []
for i in range(gm.num_policies):
    p = gm.policies[i]
    acts = [ACTION_NAMES[int(p[t, 0])] for t in range(gm.T)]
    policy_labels.append(' -> '.join(acts))

# Sort by G value
sort_idx = np.argsort(G)

fig, ax = plt.subplots(figsize=(14, 6))
colors = [COLORS['aif'] if 'cue' in policy_labels[i] else COLORS['neutral']
          for i in sort_idx]
ax.barh(range(len(G)), G[sort_idx], color=colors, alpha=0.8)
ax.set_yticks(range(len(G)))
ax.set_yticklabels([policy_labels[i] for i in sort_idx], fontsize=9)
ax.set_xlabel("Expected Free Energy G(pi) -- lower is better")
ax.set_title("EFE for All Policies (orange = visits cue)")
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nKey insight: Policies that visit the cue first have LOWER G (better).")
print("The epistemic value of visiting the cue reduces uncertainty about")
print("the reward location, making the second action more effective.")

In [ ]:
# ── Step 9: Run 20 trials and plot success rate ──────────────────────────

results = run_t_maze(
    num_trials=20,
    cue_reliability=0.9,
    gamma=4.0,
    seed=42,
    verbose=True,
)

print(f"\n=== Summary ===")
print(f"  Reward rate:     {results['reward_rate']:.1%}")
print(f"  Cue visit rate:  {results['cue_visit_rate']:.1%}")
print(f"  Mean reward:     {results['mean_reward']:.2f}")

In [ ]:
# ── Visualize trial outcomes ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Reward per trial
rewards = [r['reward'] for r in results['trial_log']]
colors_r = [COLORS['success'] if r > 0 else COLORS['danger'] if r < 0 else COLORS['neutral']
            for r in rewards]
axes[0].bar(range(len(rewards)), rewards, color=colors_r, alpha=0.8)
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Reward')
axes[0].set_title('Reward per Trial')
axes[0].axhline(y=0, color='black', linewidth=0.5)

# Plot 2: Cumulative success rate
successes = np.cumsum([1 if r > 0 else 0 for r in rewards])
rates = successes / np.arange(1, len(rewards) + 1)
axes[1].plot(rates, color=COLORS['aif'], linewidth=2)
axes[1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90% (cue reliability)')
axes[1].set_xlabel('Trial')
axes[1].set_ylabel('Cumulative Success Rate')
axes[1].set_title('Running Success Rate')
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Understanding the Dual Perspective

Let's explicitly compare how RL and Active Inference frame the same problem.

In [ ]:
dual_perspective_box(
    rl_text=(
        "The agent learns a <b>policy</b> that maximizes expected cumulative reward. "
        "Visiting the cue has no immediate reward, so the agent must learn through "
        "trial-and-error (e.g., epsilon-greedy exploration) that the cue provides "
        "useful information. This requires many episodes."
    ),
    aif_text=(
        "The agent selects policies that minimize <b>expected free energy</b>. "
        "The epistemic term in G(pi) automatically values information-gathering: "
        "visiting the cue <i>reduces uncertainty</i> about the reward location, "
        "so it has high epistemic value even with zero pragmatic value. "
        "No exploration bonus is needed -- curiosity is built in."
    ),
    title="Why does the agent visit the cue?",
)

## 4. Exercises

### Exercise 9.1: Make the Agent Prefer Punishment

Modify the C vector so the agent *prefers* punishment over reward.
What behavior do you expect? Run it and find out.

In [ ]:
# ── Exercise 9.1: Reversed preferences ──────────────────────────────────
#
# TODO: Modify C so the agent prefers punishment and avoids reward.
#       Then build a new GenerativeModel and run the agent.

C_reversed = np.zeros(NUM_OBS)
C_reversed[3] = -reward_magnitude  # reward -> now AVOIDED
C_reversed[4] = reward_magnitude   # punishment -> now PREFERRED

gm_reversed = GenerativeModel(A=[A], B=[B], C=[C_reversed], D=[D], T=2)
results_reversed = run_t_maze(
    num_trials=20,
    cue_reliability=0.9,
    gamma=4.0,
    seed=42,
    verbose=False,
)

# NOTE: run_t_maze uses its own model internally with standard C.
# To test the reversed agent, we need to run it manually:
agent_rev = ActiveInferenceAgent(gm_reversed, gamma=4.0, seed=42)
env_rev = TMazeEnv(reward_side="left", cue_reliability=0.9, seed=42)

agent_rev.reset()
obs = env_rev.reset()

print("=== Reversed-Preference Agent (prefers punishment!) ===")
for step in range(2):
    action, info = agent_rev.step([obs])
    print(f"Step {step+1}: action={ACTION_NAMES[action]}, beliefs={info['beliefs'][0].round(3)}")
    obs, reward, done = env_rev.step(action)
    print(f"         obs={OBS_NAMES[obs]}, reward={reward}, done={done}")

print(f"\nFinal location: {env_rev.location}")
print("The agent should visit the cue and then go to the WRONG arm (seeking punishment).")

### Exercise 9.2: Strong Prior for Left (Confirmation Bias)

What happens when the agent has a strong prior belief that the reward is
on the left? Does it still visit the cue, or does it go directly to the
left arm?

In [ ]:
# ── Exercise 9.2: Confirmation bias ─────────────────────────────────────
#
# Create a D vector with strong prior for reward-left.
# This models an agent that "already knows" the reward is on the left.

D_biased = np.zeros(NUM_STATES)
D_biased[0] = 0.95  # Very confident: reward is left
D_biased[1] = 0.05  # Very unlikely: reward is right

gm_biased = GenerativeModel(A=[A], B=[B], C=[C], D=[D_biased], T=2)
agent_biased = ActiveInferenceAgent(gm_biased, gamma=4.0, seed=42)

# Test on a trial where reward IS on the left
env_match = TMazeEnv(reward_side="left", cue_reliability=0.9, seed=42)
agent_biased.reset()
obs = env_match.reset()

print("=== Biased Agent (95% prior: reward is left) ===")
print("\nTrial 1: Reward IS on the left")
for step in range(2):
    action, info = agent_biased.step([obs])
    print(f"  Step {step+1}: action={ACTION_NAMES[action]}")
    obs, reward, done = env_match.step(action)
    if done:
        print(f"  Result: {'SUCCESS' if reward > 0 else 'FAILURE'}")
        break

print()

# Test on a trial where reward is on the RIGHT
env_mismatch = TMazeEnv(reward_side="right", cue_reliability=0.9, seed=42)
agent_biased.reset()
obs = env_mismatch.reset()

print("Trial 2: Reward is on the RIGHT (prior is WRONG)")
for step in range(2):
    action, info = agent_biased.step([obs])
    print(f"  Step {step+1}: action={ACTION_NAMES[action]}")
    obs, reward, done = env_mismatch.step(action)
    if done:
        print(f"  Result: {'SUCCESS' if reward > 0 else 'FAILURE'}")
        break

print("\nNotice: With a strong prior, the agent may skip the cue and go")
print("directly to the left arm. When its prior is wrong, this leads to")
print("punishment -- a form of confirmation bias!")

## Summary

In this notebook you have:

1. **Built a POMDP generative model** from scratch (A, B, C, D matrices)
2. **Visualized the model structure** through matrix heatmaps
3. **Run an Active Inference agent** on the T-maze benchmark
4. **Observed belief updating** through message passing on factor graphs
5. **Understood EFE-based action selection** and why the agent visits the cue
6. **Demonstrated near-optimal performance** over 20 trials
7. **Explored edge cases**: reversed preferences and confirmation bias

### Key Takeaways

- Active Inference agents don't need explicit exploration bonuses. The **epistemic term**
  in the Expected Free Energy naturally drives information-seeking behavior.
- The **A matrix** is the bridge between hidden states and observations. Its structure
  determines what information is available.
- **Priors (D)** shape behavior: strong priors can lead to efficient behavior when correct,
  but confirmation bias when incorrect.
- **Preferences (C)** define goals without requiring a scalar reward function.

### Next

In Module 10, we'll decompose the Expected Free Energy into its **pragmatic** and
**epistemic** components and understand exactly why curiosity emerges.